# Точная транскрипция аудио в текст (Whisper large-v3, anti-loop)

Этот ноутбук делает точную транскрипцию файлов из `audio/` и сохраняет 2 формата:
- `*.txt` — чистый текст.
- `*_timestamps.txt` — текст с таймкодами `[hh:mm:ss - hh:mm:ss]`.

Дополнительно есть пост-обработка (средняя очистка) в `*_clean.txt` и `*_timestamps_clean.txt`.


## Почему иногда появляются «повторы после 30-й минуты»

Для длинных аудио ASR-модели иногда попадают в **hallucination loop** (зацикливание текста).
Чаще это происходит при длинной непрерывной декодировке и сильной зависимости от предыдущего контекста.

В этом ноутбуке используется более устойчивый подход:
- транскрипция **чанками по 20 минут**;
- `condition_on_previous_text=False`;
- температурный fallback (`temperature=[0.0, 0.2, ...]`);
- ограничения на повторы (`repetition_penalty`, `no_repeat_ngram_size`);
- авто-проверка чанков на подозрительные повторы.


In [ ]:
# Установка зависимостей (один раз)
%pip install -q faster-whisper tqdm av nvidia-cublas-cu12 nvidia-cudnn-cu12 nvidia-cuda-runtime-cu12


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import os
import re
import site
import time

import av
import ctranslate2
from faster_whisper import WhisperModel


In [ ]:
# Для части Linux-окружений нужно явно добавить CUDA-библиотеки в LD_LIBRARY_PATH.
site_packages = Path(site.getsitepackages()[0])
cuda_lib_dirs = [
    site_packages / 'nvidia' / 'cublas' / 'lib',
    site_packages / 'nvidia' / 'cudnn' / 'lib',
    site_packages / 'nvidia' / 'cuda_runtime' / 'lib',
    site_packages / 'nvidia' / 'cuda_nvrtc' / 'lib',
]

existing_ld = os.environ.get('LD_LIBRARY_PATH', '')
extra_ld = ':'.join(str(p) for p in cuda_lib_dirs if p.exists())
if extra_ld and extra_ld not in existing_ld:
    os.environ['LD_LIBRARY_PATH'] = f'{extra_ld}:{existing_ld}' if existing_ld else extra_ld

AUDIO_DIR = Path('audio')
OUTPUT_DIR = Path('transcripts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_EXTENSIONS = {'.mp3', '.wav', '.m4a', '.flac', '.ogg', '.opus'}
audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTENSIONS])

if not audio_files:
    raise FileNotFoundError('В папке audio не найдено аудиофайлов.')

print('Найдено файлов:', len(audio_files))
for p in audio_files:
    print(' -', p)


In [ ]:
# Конфигурация качества и устойчивости
CHUNK_SECONDS = 20 * 60   # 20 минут
FORCE_LANGUAGE = 'ru'     # если нужен авто-язык: None

COMMON_PARAMS = {
    'language': FORCE_LANGUAGE,
    'task': 'transcribe',
    'vad_filter': True,
    'vad_parameters': {
        'min_silence_duration_ms': 500,
        'speech_pad_ms': 200,
    },
    'word_timestamps': False,
    'condition_on_previous_text': False,
    'compression_ratio_threshold': 2.2,
    'log_prob_threshold': -1.0,
    'no_speech_threshold': 0.6,
}

ATTEMPTS = [
    {
        'beam_size': 5,
        'best_of': 5,
        'temperature': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        'repetition_penalty': 1.05,
        'no_repeat_ngram_size': 3,
    },
    {
        'beam_size': 1,
        'best_of': 1,
        'temperature': [0.4, 0.6, 0.8, 1.0],
        'repetition_penalty': 1.2,
        'no_repeat_ngram_size': 4,
    },
]


In [ ]:
def get_duration_seconds(path: Path) -> float:
    with av.open(str(path)) as container:
        if container.duration is not None:
            return float(container.duration / av.time_base)
        durations = []
        for stream in container.streams:
            if stream.duration is not None and stream.time_base is not None:
                durations.append(float(stream.duration * stream.time_base))
        if durations:
            return max(durations)
    raise RuntimeError(f'Не удалось определить длительность: {path}')


def chunk_ranges(total_sec: float, chunk_sec: int = CHUNK_SECONDS):
    start = 0.0
    while start < total_sec:
        end = min(start + chunk_sec, total_sec)
        yield (start, end)
        start = end


def normalize_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def format_hhmmss(seconds: float) -> str:
    total_ms = int(round(seconds * 1000))
    hours, rem = divmod(total_ms, 3_600_000)
    minutes, rem = divmod(rem, 60_000)
    secs, _ = divmod(rem, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d}'


def repetition_metrics(texts: list[str]):
    if not texts:
        return {'suspicious': False, 'max_run': 0, 'dominant_ratio': 0.0, 'dominant_text': ''}

    max_run = 1
    run = 1
    prev = texts[0]
    for t in texts[1:]:
        if t == prev:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 1
            prev = t

    counts = Counter(texts)
    dominant_text, dominant_count = counts.most_common(1)[0]
    dominant_ratio = dominant_count / len(texts)

    n = len(texts)
    if n >= 80:
        suspicious = max_run >= 12 or dominant_ratio >= 0.25
    elif n >= 30:
        suspicious = max_run >= 10 or dominant_ratio >= 0.35
    else:
        suspicious = max_run >= 8

    if len(dominant_text) <= 16 and dominant_ratio >= 0.12 and n >= 20:
        suspicious = True

    return {
        'suspicious': suspicious,
        'max_run': max_run,
        'dominant_ratio': dominant_ratio,
        'dominant_text': dominant_text,
    }


In [ ]:
def transcribe_span(model: WhisperModel, audio_path: Path, start: float, end: float, attempt_idx: int):
    cfg = dict(COMMON_PARAMS)
    cfg.update(ATTEMPTS[attempt_idx])

    clip = f'{start:.3f},{end:.3f}'
    segments_iter, _ = model.transcribe(str(audio_path), clip_timestamps=clip, **cfg)

    segs = []
    for seg in segments_iter:
        text = normalize_spaces(seg.text)
        if text:
            segs.append((float(seg.start), float(seg.end), text))
    return segs


def transcribe_interval(model: WhisperModel, audio_path: Path, start: float, end: float, depth: int = 0):
    for attempt_idx in range(len(ATTEMPTS)):
        segs = transcribe_span(model, audio_path, start, end, attempt_idx)
        metrics = repetition_metrics([t for _, _, t in segs])
        if not metrics['suspicious']:
            return segs, {
                'mode': f'attempt_{attempt_idx+1}',
                'max_run': metrics['max_run'],
                'dominant_ratio': metrics['dominant_ratio'],
                'dominant_text': metrics['dominant_text'],
                'recovered_by_split': False,
            }

    span = end - start
    if depth < 2 and span > 8 * 60:
        mid = start + span / 2.0
        left, left_meta = transcribe_interval(model, audio_path, start, mid, depth + 1)
        right, right_meta = transcribe_interval(model, audio_path, mid, end, depth + 1)
        merged = left + right
        merged_metrics = repetition_metrics([t for _, _, t in merged])

        return merged, {
            'mode': f'split_depth_{depth+1}',
            'max_run': merged_metrics['max_run'],
            'dominant_ratio': merged_metrics['dominant_ratio'],
            'dominant_text': merged_metrics['dominant_text'],
            'recovered_by_split': True,
            'left_mode': left_meta.get('mode'),
            'right_mode': right_meta.get('mode'),
        }

    fallback = transcribe_span(model, audio_path, start, end, attempt_idx=1)
    metrics = repetition_metrics([t for _, _, t in fallback])
    return fallback, {
        'mode': 'forced_attempt_2',
        'max_run': metrics['max_run'],
        'dominant_ratio': metrics['dominant_ratio'],
        'dominant_text': metrics['dominant_text'],
        'recovered_by_split': False,
    }


def collapse_consecutive_identical(segments: list[tuple[float, float, str]]):
    if not segments:
        return segments

    segments = sorted(segments, key=lambda x: (x[0], x[1]))
    out = [list(segments[0])]

    for s, e, t in segments[1:]:
        prev = out[-1]
        if t == prev[2]:
            prev[1] = max(prev[1], e)
        else:
            out.append([s, e, t])

    return [(float(s), float(e), str(t)) for s, e, t in out]


def write_outputs(audio_path: Path, segments: list[tuple[float, float, str]]):
    segments = collapse_consecutive_identical(segments)

    plain_lines = [text for _, _, text in segments]
    ts_lines = [f'[{format_hhmmss(start)} - {format_hhmmss(end)}] {text}' for start, end, text in segments]

    plain_path = OUTPUT_DIR / f'{audio_path.stem}.txt'
    ts_path = OUTPUT_DIR / f'{audio_path.stem}_timestamps.txt'

    plain_path.write_text('\n'.join(plain_lines), encoding='utf-8')
    ts_path.write_text('\n'.join(ts_lines), encoding='utf-8')

    return plain_path, ts_path, len(segments)


## Запуск транскрипции


In [ ]:
device = 'cuda' if ctranslate2.get_cuda_device_count() > 0 else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'

print(f'Устройство: {device}, compute_type: {compute_type}')

model = WhisperModel(
    'large-v3',
    device=device,
    compute_type=compute_type,
    download_root='models',
)
print('Модель large-v3 загружена.')


In [ ]:
reports = []
all_started = time.perf_counter()

for audio_path in audio_files:
    print(f'\n=== {audio_path.name} ===')
    file_started = time.perf_counter()

    total_sec = get_duration_seconds(audio_path)
    chunks = list(chunk_ranges(total_sec, chunk_sec=CHUNK_SECONDS))
    print(f'Длительность: {total_sec/60:.2f} мин | чанков: {len(chunks)}')

    collected_segments = []
    split_recoveries = 0

    for idx, (start, end) in enumerate(chunks, start=1):
        segs, meta = transcribe_interval(model, audio_path, start, end, depth=0)
        collected_segments.extend(segs)
        if meta.get('recovered_by_split'):
            split_recoveries += 1

        print(
            f'  chunk {idx:02d}/{len(chunks)} '
            f'[{start/60:.1f}-{end/60:.1f} мин] '
            f'segs={len(segs)} mode={meta.get("mode")} '
            f'max_run={meta.get("max_run")} dom={meta.get("dominant_ratio", 0):.2f}'
        )

    plain_path, ts_path, final_segments = write_outputs(audio_path, collected_segments)

    elapsed = time.perf_counter() - file_started
    report = {
        'file': str(audio_path),
        'duration_min': round(total_sec / 60, 2),
        'chunks': len(chunks),
        'split_recoveries': split_recoveries,
        'segments': final_segments,
        'elapsed_min': round(elapsed / 60, 2),
        'txt': str(plain_path),
        'txt_timestamps': str(ts_path),
    }
    reports.append(report)

    print(
        f'Готово: {audio_path.name} | сегментов={final_segments} | '
        f'чанков={len(chunks)} | split_recoveries={split_recoveries} | '
        f'время={report["elapsed_min"]} мин'
    )

print(f'\nОбщее время: {(time.perf_counter() - all_started)/60:.2f} мин')


## Пост-обработка (средняя очистка)

Этот шаг делает отдельные файлы:
- `*_clean.txt`
- `*_timestamps_clean.txt`

Исходные транскрипты не перезаписываются.


In [ ]:
WORD_RE = r'[A-Za-zА-Яа-яЁё0-9-]+'
DUP_WORD_RE = re.compile(rf'\b({WORD_RE})\b(?:\s+\1\b)+', flags=re.IGNORECASE)
DUP_WORD_COMMA_RE = re.compile(rf'\b({WORD_RE})\b\s*,\s*\1\b', flags=re.IGNORECASE)
DUP_PHRASE_RE = re.compile(
    rf'\b((?:{WORD_RE}\s+){{1,3}}{WORD_RE})\b(?:\s*,?\s*\1\b)+',
    flags=re.IGNORECASE,
)
TS_LINE_RE = re.compile(r'^\[(\d{2}:\d{2}:\d{2}) - (\d{2}:\d{2}:\d{2})\]\s*(.*)$')

MEDIUM_FILLERS = [
    re.compile(r'\b(ээ+|эм+|мм+|м-м|а-а)\b', flags=re.IGNORECASE),
    re.compile(r'\bкак\s+бы\b', flags=re.IGNORECASE),
    re.compile(r'\b(ну|типа|короче|блин)\b', flags=re.IGNORECASE),
    re.compile(r'\b(в\s+общем(?:-то)?|это\s+самое|так\s+сказать)\b', flags=re.IGNORECASE),
    re.compile(r'(,\s*)вот(?=(\s*,|\s|$))', flags=re.IGNORECASE),
]


def normalize_punctuation(text: str) -> str:
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    text = re.sub(r'([.!?])\s*,\s*', r'\1 ', text)
    text = re.sub(r',\s*([.!?])', r'\1', text)
    text = re.sub(r'(,\s*){2,}', ', ', text)
    text = re.sub(r"([,.;:!?])(?![\s\"')\]»]|$)", r'\1 ', text)
    text = re.sub(r'([,.;:!?])\1+', r'\1', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^[,;:.!?\-—\s]+', '', text)
    text = re.sub(r'([,\-—]\s*)+$', '', text)
    text = re.sub(
        r'([.!?]\s+)([а-яёa-z])',
        lambda m: m.group(1) + m.group(2).upper(),
        text,
        flags=re.IGNORECASE,
    )
    return text.strip()


def clean_line_medium(text: str) -> str:
    cleaned = text

    for _ in range(3):
        cleaned = DUP_WORD_RE.sub(r'\1', cleaned)
        cleaned = DUP_WORD_COMMA_RE.sub(r'\1', cleaned)

    for _ in range(2):
        cleaned = DUP_PHRASE_RE.sub(r'\1', cleaned)

    for p in MEDIUM_FILLERS:
        if p.pattern.startswith('(,\\s*)вот'):
            cleaned = p.sub(r'\1', cleaned)
        else:
            cleaned = p.sub('', cleaned)

    cleaned = normalize_punctuation(cleaned)
    for _ in range(2):
        cleaned = DUP_WORD_COMMA_RE.sub(r'\1', cleaned)
    cleaned = normalize_punctuation(cleaned)

    if cleaned and text[:1].isupper() and cleaned[:1].islower():
        cleaned = cleaned[:1].upper() + cleaned[1:]

    return cleaned


def write_clean_versions(txt_path: Path, ts_path: Path):
    txt_clean = txt_path.with_name(f'{txt_path.stem}_clean{txt_path.suffix}')
    ts_clean = ts_path.with_name(f'{ts_path.stem}_clean{ts_path.suffix}')

    txt_lines = []
    ts_lines = []

    for raw in ts_path.read_text(encoding='utf-8').splitlines():
        m = TS_LINE_RE.match(raw)
        if not m:
            continue
        start_ts, end_ts, text = m.groups()
        cl = clean_line_medium(text)
        if cl:
            txt_lines.append(cl)
            ts_lines.append(f'[{start_ts} - {end_ts}] {cl}')

    txt_clean.write_text('\n'.join(txt_lines), encoding='utf-8')
    ts_clean.write_text('\n'.join(ts_lines), encoding='utf-8')

    return txt_clean, ts_clean


clean_reports = []
for r in reports:
    txt_clean, ts_clean = write_clean_versions(Path(r['txt']), Path(r['txt_timestamps']))
    clean_reports.append({
        'source_txt': r['txt'],
        'source_ts': r['txt_timestamps'],
        'clean_txt': str(txt_clean),
        'clean_ts': str(ts_clean),
    })

print('Готово. Созданы clean-версии:')
for c in clean_reports:
    print(' -', c['clean_txt'])
    print('   ', c['clean_ts'])


In [ ]:
print('Итог:')
for r in reports:
    print('\n', r['file'])
    print('  txt:', r['txt'])
    print('  txt+timestamps:', r['txt_timestamps'])
    print('  chunks:', r['chunks'], '| segments:', r['segments'], '| elapsed_min:', r['elapsed_min'])
